# 0. 시작하기

이 실습에서 다루는 질문은 하나입니다.

> **AI가 고친 코드가 기존 동작을 지켰는지, 무엇으로 어떻게 확인하는가?**

| 노트북 | 질문 | 확인 도구 |
|---|---|---|
| A. 변경 영향 | 이 변경이 어디까지 영향을 주는가? 도구가 놓치는 곳은? | 동작 기록 비교, def-use 영향 분석 |
| B. Dafny 모델 | AI가 만든 Dafny 모델을 믿어도 되는가? | 계약 읽기, Dafny 검증, 차분 테스트 |
| C. 변경 요청 1 | 테스트를 다 통과한 AI 패치 2개 중 무엇을 내보낼 수 있는가? | 위 도구를 모두 써서 판단 |
| D. 재검증 범위 | 코드가 바뀔 때마다 증명을 전부 다시 해야 하는가? | 바뀐 것의 종류로 재검증 범위 줄이기 |
| E. 정리 규칙 | 코드 정리 규칙을 일괄 적용해도 동작이 그대로인가? | 규칙의 조건 적기, 테스트, 동작 기록 비교 |

### 노트북 쓰는 법

- 회색 칸(코드 셀)을 클릭하고 **Shift+Enter**로 실행합니다. 결과가 바로 아래에 나옵니다.
- **위에서부터 차례로** 실행하세요. 노트북을 새로 열었으면 맨 위의 `import lab` 셀부터 다시 실행합니다.
- 실행 중에는 셀 왼쪽에 `[*]`가 표시됩니다. Dafny 증명은 1~2분 걸릴 수 있습니다.
- 결과 보고서 안의 `+` 표시는 눌러서 펼칠 수 있습니다. 보고서 파일은 `out/html/`에도 저장되어 브라우저에서 따로 열 수 있습니다.

In [1]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## 대상 시스템: HomeHub

HomeHub는 이 실습을 위해 만든 **가상의 가전 허브 프로그램**입니다. 세탁기, 냉장고, 에어컨, 누수센서 같은 가전이 보내는 메시지를 받아 처리합니다. 2014~2017년에 여러 사람이 고쳐 온 Java 8 코드라는 설정입니다(30개 타입, 약 1,600줄, 테스트 38개).

아래 셀을 실행하면 **HomeHub 구조 지도**가 나옵니다. 지도를 확대하듯 봅니다. **⊕ 표시가 있는 상자를 누르면** 그 안의 자세한 그림이 나오고, 위의 경로나 **← 밖으로**를 누르면 나옵니다.

처음에는 이 순서로 봅니다(약 15분).
1. 바깥 그림: HomeHub가 무엇을 받고 무엇을 남기는지
2. 가운데 HomeHub 상자를 눌러 안의 부품 보기
3. **▶ 메시지 하나 따라가기**: 메시지 한 줄이 어느 부품을 거쳐 처리되는지 한 단계씩
4. 궁금한 부품(특히 **대기열**, **접수·꺼내기**)을 눌러 보기

다 외울 필요는 없습니다. 실습 중에 낯선 부품이 나오면 그때 다시 띄워 보면 됩니다. "새 탭에서 열기"로 띄워 두면 편합니다.

In [ ]:
lab.structure()   # HomeHub 구조 지도 (파일: docs/homehub_구조.html)

### 바꾸면 안 되는 규칙 R1~R4 (대기열이 가득 찼을 때)

| 번호 | 규칙 |
|---|---|
| R1 | 대기 이벤트 수는 정해진 상한을 넘지 않는다 |
| R2 | 누수·과열 같은 중요(critical) 이벤트는 절대 거부하지 않는다 |
| R3 | 대기열에 남는 이벤트의 순서는 바뀌지 않는다. 자리를 만들어야 하면 가장 오래된 비중요 이벤트를 밀어낸다 |
| R4 | 비중요 이벤트는 대기열이 가득 차 있고 autoDrain 설정이 꺼져 있을 때만 거부한다 |

### 변경 요청 2건

| 이름 | 폴더 | 요청 내용 |
|---|---|---|
| 변경 요청 1: 대기열 늘리기 | `scenarios/req1` | 저녁에 이벤트가 몰리면 거부된다. 최대 32개까지 받게 해 달라 |
| 변경 요청 2: 대기열 연산 빠르게 | `scenarios/req2` | 허브 칩에서 `%` 연산이 느리다. 대기열 코드에서 `%`를 없애 달라(동작은 그대로) |

각 폴더에는 AI가 만든 Java 패치(`ai*/`)와 Dafny 모델(`model/`)이 있습니다. 패치 폴더에는 바뀐 파일만 있고, 실행할 때 레거시 전체를 복사한 뒤 덮어써서 빌드합니다.

## 환경 확인

도구 버전을 확인하고, 기존 테스트 38개를 돌립니다.

In [2]:
lab.run("check-env")

$ make check-env
java : openjdk version "21.0.12.1" 2026-08-18


javac: javac 21.0.12.1


py   : Python 3.13.16


dafny: 4.11.0+fcb2042d6d043a2634f0854338c08feeaaaf4ae2


jupyter: jupyterlab       : 4.6.4


codex: codex-cli 0.162.1 · Not logged in


In [3]:
lab.run("test")

$ make test


tests: 38, passed: 38, failed: 0


## Dafny 증명 전체 점검 (1~2분)

이 실습에서 통과해야 하는 Dafny 증명을 모두 돌립니다. **모든 항목이 "검증 통과 / 예상대로"여야 합니다.** 시간 초과(timed out)가 나오면 강사에게 알려 주세요.

In [4]:
lab.run("verify-all")

$ make verify-all


Dafny dfyconfig.toml                           검증 71개, 오류 0개


Dafny Types.dfy RingQueue.dfy Dispatcher.dfy P 검증 73개, 오류 0개


Dafny Types.dfy RingQueue.dfy Dispatcher.dfy P 검증 71개, 오류 0개


Dafny RewriteRules.dfy                         검증 19개, 오류 0개
